In [2]:
import warnings
import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score, davies_bouldin_score
from hdbscan import HDBSCAN, approximate_predict
from prince import FAMD
from IPython.display import display

In [3]:
df = pd.read_csv("C:/Users/WELCOME/OneDrive/Desktop/ML 7641/ML/encoded_joined_fixed.csv")
df = df.sort_values("start_interval").reset_index(drop=True)

display(df.head(10))
print("shape: ", df.shape)
print("columns: ", df.columns)

,start_interval,route_1_0,route_1_1,route_2_0,route_2_1,route_3_0,route_3_1,route_4_0,route_4_1,route_5_0,...,bus,motorcycle,taxi,brooklyn,east_60,fdr,nj,queens,west_60,west_side_hwy
0,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,True,False,False,True,False,False,False,False,False,False
1,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,True,False,False,False,True,False,False,False,False,False
2,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,True,False,False,False,False,False,True,False,False,False
3,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,False,False,True,False,False,False,False,False,False
4,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,False,False,False,False,False,True,False,False,False
5,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,False,False,False,False,False,False,True,False,False
6,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,False,True
7,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,True,False
8,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,False,True,False,False,False,False,False,False,True
9,2025-08-01 00:00:00,False,True,False,True,True,True,True,True,True,...,False,False,False,False,False,False,False,False,True,False


shape:  (393120, 87)
columns:  Index(['start_interval', 'route_1_0', 'route_1_1', 'route_2_0', 'route_2_1',
       'route_3_0', 'route_3_1', 'route_4_0', 'route_4_1', 'route_5_0',
       'route_5_1', 'route_6_0', 'route_6_1', 'route_6X_0', 'route_6X_1',
       'route_7_0', 'route_7_1', 'route_7X_0', 'route_7X_1', 'route_A_0',
       'route_A_1', 'route_B_0', 'route_B_1', 'route_C_0', 'route_C_1',
       'route_D_0', 'route_D_1', 'route_E_0', 'route_E_1', 'route_F_0',
       'route_F_1', 'route_FS_0', 'route_FS_1', 'route_FX_0', 'route_FX_1',
       'route_G_0', 'route_G_1', 'route_GS_0', 'route_GS_1', 'route_H_0',
       'route_H_1', 'route_J_0', 'route_J_1', 'route_L_0', 'route_L_1',
       'route_M_0', 'route_M_1', 'route_N_0', 'route_N_1', 'route_Q_0',
       'route_Q_1', 'route_R_0', 'route_R_1', 'route_SI_0', 'route_SI_1',
       'route_SS_0', 'route_SS_1', 'route_W_0', 'route_W_1', 'route_Z_0',
       'route_Z_1', 'total_trips', 'avg_delay', 'entries',
       'excluded_roadway_en

In [4]:
target_cols = ["entries", "excluded_roadway_entries"]
id_col = "start_interval"
cont_cols = ["total_trips", "avg_delay"]
binary_cols = [c for c in df.columns if c not in target_cols + cont_cols + [id_col]]

df[binary_cols] = df[binary_cols].astype(int)

display(df.head())
print(f"binary features: {len(binary_cols)}, continuous features: {len(cont_cols)}")

,start_interval,route_1_0,route_1_1,route_2_0,route_2_1,route_3_0,route_3_1,route_4_0,route_4_1,route_5_0,...,bus,motorcycle,taxi,brooklyn,east_60,fdr,nj,queens,west_60,west_side_hwy
0,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,1,0,0,1,0,0,0,0,0,0
1,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,1,0,0,0,1,0,0,0,0,0
2,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,1,0,0,0,0,0,1,0,0,0
3,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,0,0,0,1,0,0,0,0,0,0
4,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,0,0,0,0,0,0,1,0,0,0


binary features: 82, continuous features: 2


In [5]:
intervals = df["start_interval"].unique()
intervals = np.sort(intervals)

n_int = len(intervals)
train_end = int(n_int * 0.7)
val_end   = int(n_int * 0.85)

train_intervals = set(intervals[:train_end])
val_intervals   = set(intervals[train_end:val_end])
test_intervals  = set(intervals[val_end:])

df_train = df[df["start_interval"].isin(train_intervals)].copy()
df_val   = df[df["start_interval"].isin(val_intervals)].copy()
df_test  = df[df["start_interval"].isin(test_intervals)].copy()

print("Train ∩ Val:", len(set(df_train["start_interval"]) & set(df_val["start_interval"])))
print("Val ∩ Test:", len(set(df_val["start_interval"]) & set(df_test["start_interval"])))
print("Train ∩ Test:", len(set(df_train["start_interval"]) & set(df_test["start_interval"])))

Train ∩ Val: 0
Val ∩ Test: 0
Train ∩ Test: 0


In [6]:
X_train = df_train[cont_cols + binary_cols]
X_val   = df_val[cont_cols + binary_cols]
X_test  = df_test[cont_cols + binary_cols]

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (275184, 84), Val: (58968, 84), Test: (58968, 84)


In [7]:
sample_size = min(10000, len(X_train))
X_sample = X_train.sample(n=sample_size, random_state=42)

param_grid = {
    "famd__n_components": [8, 10, 12],
    "hdbscan__min_cluster_size": [30, 50],
    "hdbscan__min_samples": [None, 10, 20],
}

In [8]:
def run_pipeline(X_df, n_comp, min_cluster_size, min_samples):
    famd = FAMD(n_components=n_comp, random_state=42)
    X_famd = famd.fit_transform(X_df)

    clusterer = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean"
    )
    labels = clusterer.fit_predict(X_famd)

    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels[unique_labels != -1])

    noise_pct = (labels == -1).mean() * 100

    mask = labels != -1
    if mask.sum() > 10 and len(np.unique(labels[mask])) > 1:
        sil = silhouette_score(X_famd[mask], labels[mask])
        dbi = davies_bouldin_score(X_famd[mask], labels[mask])
    else:
        sil, dbi = np.nan, np.nan

    return {
        "famd": famd,
        "clusterer": clusterer,
        "n_clusters": n_clusters,
        "noise_pct": noise_pct,
        "silhouette": sil,
        "dbi": dbi
    }

In [9]:
warnings.simplefilter(action='ignore', category=FutureWarning)
results = []
for n_comp in param_grid["famd__n_components"]:
    for mcs in param_grid["hdbscan__min_cluster_size"]:
        for ms in param_grid["hdbscan__min_samples"]:
            print(f"\nRunning n_components={n_comp}, min_cluster_size={mcs}, min_samples={ms}")
            res = run_pipeline(X_sample, n_comp, mcs, ms)
            results.append({
                "n_components": n_comp,
                "min_cluster_size": mcs,
                "min_samples": ms,
                "n_clusters": res["n_clusters"],
                "noise_pct": res["noise_pct"],
                "silhouette": res["silhouette"],
                "dbi": res["dbi"],
                "model": res
            })

df_results = pd.DataFrame(results)
df_results = df_results.sort_values(["dbi", "silhouette"], ascending=[True, False])
best = df_results.iloc[0]

print(best[["n_components", "min_cluster_size", "silhouette", "dbi"]])

best_params = {
    "n_components": int(best["n_components"]),
    "min_cluster_size": int(best["min_cluster_size"]),
    "min_samples": None if pd.isna(best["min_samples"]) else (
        int(best["min_samples"]) if best["min_samples"] is not None else None
    )
}


Running n_components=8, min_cluster_size=30, min_samples=None

Running n_components=8, min_cluster_size=30, min_samples=10

Running n_components=8, min_cluster_size=30, min_samples=20

Running n_components=8, min_cluster_size=50, min_samples=None

Running n_components=8, min_cluster_size=50, min_samples=10

Running n_components=8, min_cluster_size=50, min_samples=20

Running n_components=10, min_cluster_size=30, min_samples=None

Running n_components=10, min_cluster_size=30, min_samples=10

Running n_components=10, min_cluster_size=30, min_samples=20

Running n_components=10, min_cluster_size=50, min_samples=None

Running n_components=10, min_cluster_size=50, min_samples=10

Running n_components=10, min_cluster_size=50, min_samples=20

Running n_components=12, min_cluster_size=30, min_samples=None

Running n_components=12, min_cluster_size=30, min_samples=10

Running n_components=12, min_cluster_size=30, min_samples=20

Running n_components=12, min_cluster_size=50, min_samples=None

R

In [10]:
display(df_results)

,n_components,min_cluster_size,min_samples,n_clusters,noise_pct,silhouette,dbi,model
6,10,30,NaN,3,22.33,0.518574,0.657569,"{'famd': FAMD(n_components=10, random_state=42..."
5,8,50,20.0,4,19.34,0.464237,0.676789,"{'famd': FAMD(n_components=8, random_state=42)..."
3,8,50,NaN,5,49.24,0.507578,0.694425,"{'famd': FAMD(n_components=8, random_state=42)..."
0,8,30,NaN,5,23.04,0.289963,0.707624,"{'famd': FAMD(n_components=8, random_state=42)..."
11,10,50,20.0,4,25.39,0.448781,0.714560,"{'famd': FAMD(n_components=10, random_state=42..."
15,12,50,NaN,3,21.25,0.493348,0.728657,"{'famd': FAMD(n_components=12, random_state=42..."
12,12,30,NaN,3,19.59,0.482426,0.746352,"{'famd': FAMD(n_components=12, random_state=42..."
17,12,50,20.0,3,17.68,0.471432,0.768686,"{'famd': FAMD(n_components=12, random_state=42..."
9,10,50,NaN,2,6.91,0.434489,0.770237,"{'famd': FAMD(n_components=10, random_state=42..."
16,12,50,10.0,3,16.23,0.462015,0.791375,"{'famd': FAMD(n_components=12, random_state=42..."


In [19]:
#2 n_components 8 min_cluster_size 30 min_samples 20 n_cluster = 9

famd_final = FAMD(n_components=8, random_state=42)
X_train_famd = famd_final.fit_transform(X_train)

clusterer_final = HDBSCAN(
    min_cluster_size=30,
    min_samples=20,
    metric="euclidean",
    prediction_data=True
)
clusterer_final.fit(X_train_famd)

labels_train = clusterer_final.labels_
df_train["cluster"] = labels_train

display(df_train.head())

,start_interval,route_1_0,route_1_1,route_2_0,route_2_1,route_3_0,route_3_1,route_4_0,route_4_1,route_5_0,...,motorcycle,taxi,brooklyn,east_60,fdr,nj,queens,west_60,west_side_hwy,cluster
0,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,0,0,1,0,0,0,0,0,0,1082
1,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,0,0,0,1,0,0,0,0,0,1082
2,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,0,0,0,0,0,1,0,0,0,1082
3,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,0,0,1,0,0,0,0,0,0,1082
4,2025-08-01 00:00:00,0,1,0,1,1,1,1,1,1,...,0,0,0,0,0,1,0,0,0,1082


In [20]:
X_val_famd = famd_final.transform(X_val)
X_test_famd = famd_final.transform(X_test)

labels_val, probs_val = approximate_predict(clusterer_final, X_val_famd)
labels_test, probs_test = approximate_predict(clusterer_final, X_test_famd)

df_val["cluster"] = labels_val
df_test["cluster"] = labels_test

display(df_val.head())
display(df_test.head())

,start_interval,route_1_0,route_1_1,route_2_0,route_2_1,route_3_0,route_3_1,route_4_0,route_4_1,route_5_0,...,motorcycle,taxi,brooklyn,east_60,fdr,nj,queens,west_60,west_side_hwy,cluster
275184,2025-09-15 12:00:00,1,1,1,1,1,1,1,1,1,...,0,1,0,0,0,1,0,0,0,-1
275185,2025-09-15 12:00:00,1,1,1,1,1,1,1,1,1,...,0,1,0,0,0,0,0,0,1,-1
275186,2025-09-15 12:00:00,1,1,1,1,1,1,1,1,1,...,0,0,1,0,0,0,0,0,0,-1
275187,2025-09-15 12:00:00,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,1,0,-1
275188,2025-09-15 12:00:00,1,1,1,1,1,1,1,1,1,...,0,0,0,1,0,0,0,0,0,-1


,start_interval,route_1_0,route_1_1,route_2_0,route_2_1,route_3_0,route_3_1,route_4_0,route_4_1,route_5_0,...,motorcycle,taxi,brooklyn,east_60,fdr,nj,queens,west_60,west_side_hwy,cluster
334152,2025-09-25 06:00:00,1,1,1,1,0,1,1,1,1,...,0,1,0,0,0,0,0,0,1,-1
334153,2025-09-25 06:00:00,1,1,1,1,0,1,1,1,1,...,0,0,0,0,0,0,1,0,0,-1
334154,2025-09-25 06:00:00,1,1,1,1,0,1,1,1,1,...,0,0,1,0,0,0,0,0,0,-1
334155,2025-09-25 06:00:00,1,1,1,1,0,1,1,1,1,...,0,0,0,0,0,0,0,1,0,-1
334156,2025-09-25 06:00:00,1,1,1,1,0,1,1,1,1,...,0,0,0,1,0,0,0,0,0,-1


In [21]:
mask = labels_train != -1
if mask.sum() > 10:
    sil_final = silhouette_score(X_train_famd[mask], labels_train[mask])
    dbi_final = davies_bouldin_score(X_train_famd[mask], labels_train[mask])
else:
    sil_final, dbi_final = np.nan, np.nan

print(f"Silhouette: {sil_final:.4f}")
print(f"Davies–Bouldin: {dbi_final:.4f}")
print(f"Clusters: {len(np.unique(labels_train)) - (1 if -1 in labels_train else 0)}")
print(f"Noise: {(labels_train == -1).mean() * 100:.2f}%")

Silhouette: 0.3486
Davies–Bouldin: 0.9145
Clusters: 3138
Noise: 27.00%


In [22]:
#
df_train.to_csv("C:/Users/WELCOME/OneDrive/Desktop/ML 7641/ML/train_clusters.csv", index=False)
df_val.to_csv("C:/Users/WELCOME/OneDrive/Desktop/ML 7641/ML/val_clusters.csv", index=False)
df_test.to_csv("C:/Users/WELCOME/OneDrive/Desktop/ML 7641/ML/test_clusters.csv", index=False)

In [23]:
#
val_df = pd.read_csv("C:/Users/WELCOME/OneDrive/Desktop/ML 7641/ML/train_clusters.csv")
train_df = pd.read_csv("C:/Users/WELCOME/OneDrive/Desktop/ML 7641/ML/val_clusters.csv")
test_df = pd.read_csv("C:/Users/WELCOME/OneDrive/Desktop/ML 7641/ML/test_clusters.csv")

In [24]:
#
val_df['cluster'] = val_df['cluster'].astype(int)
print("Validation Set Cluster Counts:")
print(val_df['cluster'].value_counts().sort_index())
print("\nValidation Set Cluster Percentages:")
print((val_df['cluster'].value_counts(normalize=True).sort_index() * 100))

Validation Set Cluster Counts:
cluster
-1       74290
 0          42
 1          42
 2          84
 3          42
         ...  
 3133       41
 3134      107
 3135      342
 3136      132
 3137       49
Name: count, Length: 3139, dtype: int64

Validation Set Cluster Percentages:
cluster
-1       26.996482
 0        0.015263
 1        0.015263
 2        0.030525
 3        0.015263
           ...    
 3133     0.014899
 3134     0.038883
 3135     0.124280
 3136     0.047968
 3137     0.017806
Name: proportion, Length: 3139, dtype: float64
